# Phase 4A: Bucketing & Segmentation in PySpark
This comprehensive notebook covers the practice tasks for dividing continuous values into categories (bucketing/segmentation) and includes the reflection answers at the end.

## Environment Setup & Data Generation
We first install PySpark and generate the local CSV files required for this exercise so that it runs perfectly in Google Colab.

In [1]:
!pip install -q pyspark

import os
customers_data = '''customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
2,Emma,Jones,emma.jones@webmail.com,555-0002,456 Oak St,Centerville,OH,45459
3,Olivia,Brown,olivia.brown@outlook.com,555-0003,789 Pine St,Greenville,SC,29601
4,Liam,Johnson,liam.johnson@gmail.com,555-0004,101 Maple St,Riverside,CA,92501
5,Noah,Williams,noah.williams@yahoo.com,555-0005,202 Birch St,Lakeside,TX,75001
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
7,Isabella,Davis,isabella.davis@icloud.com,555-0007,404 Spruce St,Boise,ID,83701
8,James,Martinez,james.martinez@live.com,555-0008,505 Walnut St,Des Moines,IA,50301
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201
11,Mia,Lopez,mia.lopez@mail.com,555-0011,808 Oak St,Miami,FL,33101
12,William,Anderson,william.anderson@fastmail.com,555-0012,909 Pine St,Nashville,TN,37201
13,Amelia,Thomas,amelia.thomas@protonmail.com,555-0013,1010 Elm St,Denver,CO,80201
14,Ethan,Taylor,ethan.taylor@inbox.com,555-0014,1111 Birch St,Minneapolis,MN,55401
15,Harper,Jackson,harper.jackson@outlook.com,555-0015,1212 Cedar St,Seattle,WA,98101
16,Jackson,White,jackson.white@ymail.com,555-0016,1313 Spruce St,Atlanta,GA,30301
17,Charlotte,Harris,charlotte.harris@webmail.com,555-0017,1414 Walnut St,San Diego,CA,92101
18,Oliver,Martin,oliver.martin@icloud.com,555-0018,1515 Cherry St,Indianapolis,IN,46201
19,Madison,Thompson,madison.thompson@gmail.com,555-0019,1616 Maple St,Charlotte,NC,28201
20,Elijah,Garcia,elijah.garcia@zoho.com,555-0020,1717 Oak St,Detroit,MI,48201'''

sales_data = '''sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,1,1,15-01-2024,2,39.98
2,1,3,20-01-2024,1,29.99
3,2,2,16-01-2024,1,25
4,2,4,22-01-2024,3,89.97
5,3,5,17-01-2024,2,49.98
6,4,6,18-01-2024,4,119.96
7,4,7,25-01-2024,1,15.5
8,5,8,19-01-2024,3,66.75
9,6,9,20-01-2024,2,40
10,7,10,21-01-2024,5,110.95
11,8,11,22-01-2024,1,20
12,9,12,23-01-2024,4,79.96
13,10,13,24-01-2024,2,55
14,11,14,25-01-2024,1,25
15,12,15,26-01-2024,3,67.47
16,13,16,27-01-2024,2,34
17,14,17,28-01-2024,1,15
18,15,18,29-01-2024,4,92
19,16,19,30-01-2024,3,60
20,17,20,31-01-2024,2,40
21,18,1,01-02-2024,1,19.99
22,18,2,05-02-2024,2,39.98
23,19,3,06-02-2024,1,29.99
24,20,4,07-02-2024,3,89.97
25,21,5,08-02-2024,2,49.98'''

with open('customers.csv', 'w') as f: f.write(customers_data)
with open('sales.csv', 'w') as f: f.write(sales_data)
print('Data files generated successfully!')


Data files generated successfully!


## Base Data Preparation
Before we can segment customers by their spend, we need to load the data, clean it, and aggregate total spend per customer.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when

spark = SparkSession.builder.appName("Phase4A-Bucketing").getOrCreate()

# Load and clean data
customers_df = spark.read.csv('customers.csv', header=True, inferSchema=True).dropna(subset=["customer_id"])
sales_df = spark.read.csv('sales.csv', header=True, inferSchema=True).dropna(subset=["customer_id"])

# Aggregate sales to get total spend per customer
customer_spend = sales_df.groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spend")) \
    .join(customers_df, "customer_id", "inner") \
    .select("customer_id", "first_name", "last_name", "total_spend")

print("Base Data: Customer Spend")
customer_spend.show(5)


Base Data: Customer Spend
+-----------+----------+---------+------------------+
|customer_id|first_name|last_name|       total_spend|
+-----------+----------+---------+------------------+
|          1|      John|    Smith|             69.97|
|          2|      Emma|    Jones|            114.97|
|          3|    Olivia|    Brown|             49.98|
|          4|      Liam|  Johnson|135.45999999999998|
|          5|      Noah| Williams|             66.75|
+-----------+----------+---------+------------------+
only showing top 5 rows


## Task 1: Create Gold/Silver/Bronze segmentation using conditional logic
Using PySpark's `when().otherwise()` function, we will assign fixed logical rules to group customers into Gold (>$10k), Silver ($5k-$10k), and Bronze (<$5k).
*Note: Because our sample dataset contains very small amounts, most (if not all) customers will fall into the Bronze tier based on these fixed business rules.*

In [3]:
segment_cond_df = customer_spend.withColumn(
    "segment_fixed",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

segment_cond_df.show(5)


+-----------+----------+---------+------------------+-------------+
|customer_id|first_name|last_name|       total_spend|segment_fixed|
+-----------+----------+---------+------------------+-------------+
|          1|      John|    Smith|             69.97|       Bronze|
|          2|      Emma|    Jones|            114.97|       Bronze|
|          3|    Olivia|    Brown|             49.98|       Bronze|
|          4|      Liam|  Johnson|135.45999999999998|       Bronze|
|          5|      Noah| Williams|             66.75|       Bronze|
+-----------+----------+---------+------------------+-------------+
only showing top 5 rows


## Task 2: Group data by segment and count customers
We can aggregate the segmented DataFrame to see how many customers fall into each predefined bucket.

In [4]:
segment_counts = segment_cond_df.groupBy("segment_fixed").count()
segment_counts.show()


+-------------+-----+
|segment_fixed|count|
+-------------+-----+
|       Bronze|   20|
+-------------+-----+



## Task 3: Try quantile-based segmentation
Unlike fixed conditional logic, Quantiles are dynamic. Using `approxQuantile`, we find the 33rd and 66th percentile values. This ensures our customers are divided roughly equally into three tiers (Top 33%, Middle 33%, Bottom 33%) regardless of what the actual dollar amounts are.

In [5]:
quantiles = segment_cond_df.approxQuantile("total_spend", [0.33, 0.66], 0)
print(f"Calculated Quantiles (33rd & 66th percentiles): {quantiles}")

q1 = quantiles[0]
q2 = quantiles[1]

final_df = segment_cond_df.withColumn(
    "segment_quantile",
    when(col("total_spend") > q2, "Gold (Top 33%)")
    .when((col("total_spend") > q1) & (col("total_spend") <= q2), "Silver (Middle 33%)")
    .otherwise("Bronze (Bottom 33%)")
)

final_df.show(5)


Calculated Quantiles (33rd & 66th percentiles): [40.0, 69.97]
+-----------+----------+---------+------------------+-------------+-------------------+
|customer_id|first_name|last_name|       total_spend|segment_fixed|   segment_quantile|
+-----------+----------+---------+------------------+-------------+-------------------+
|          1|      John|    Smith|             69.97|       Bronze|Silver (Middle 33%)|
|          2|      Emma|    Jones|            114.97|       Bronze|     Gold (Top 33%)|
|          3|    Olivia|    Brown|             49.98|       Bronze|Silver (Middle 33%)|
|          4|      Liam|  Johnson|135.45999999999998|       Bronze|     Gold (Top 33%)|
|          5|      Noah| Williams|             66.75|       Bronze|Silver (Middle 33%)|
+-----------+----------+---------+------------------+-------------+-------------------+
only showing top 5 rows


## Task 4: Compare results of different methods
Here we will view both segmentation columns side by side. Notice how the fixed logic labels everyone as Bronze, but the quantile approach accurately categorizes them based on their relative spend.

In [6]:
final_df.select("customer_id", "total_spend", "segment_fixed", "segment_quantile") \
    .orderBy("total_spend", ascending=False).show(15)

print("Customer Count by Quantile Segment:")
final_df.groupBy("segment_quantile").count().orderBy("segment_quantile").show()


+-----------+------------------+-------------+-------------------+
|customer_id|       total_spend|segment_fixed|   segment_quantile|
+-----------+------------------+-------------+-------------------+
|          4|135.45999999999998|       Bronze|     Gold (Top 33%)|
|          2|            114.97|       Bronze|     Gold (Top 33%)|
|          7|            110.95|       Bronze|     Gold (Top 33%)|
|         15|              92.0|       Bronze|     Gold (Top 33%)|
|         20|             89.97|       Bronze|     Gold (Top 33%)|
|          9|             79.96|       Bronze|     Gold (Top 33%)|
|          1|             69.97|       Bronze|Silver (Middle 33%)|
|         12|             67.47|       Bronze|Silver (Middle 33%)|
|          5|             66.75|       Bronze|Silver (Middle 33%)|
|         16|              60.0|       Bronze|Silver (Middle 33%)|
|         18|             59.97|       Bronze|Silver (Middle 33%)|
|         10|              55.0|       Bronze|Silver (Middle 3

## Task 5: Reflection Questions & Answers

**1. Why do we convert continuous values into categories?**
Converting continuous data (like exact dollar amounts) into categories (like Gold, Silver, Bronze) makes it much easier to digest for business stakeholders. It simplifies analysis, enables targeted marketing (e.g., sending an email only to "Gold" members), and reduces noise by grouping similar behaviors together instead of dealing with infinite unique continuous values.

**2. What is the difference between business segmentation and technical bucketing?**
* **Business Segmentation:** Driven by business logic or marketing goals. For example, a marketing team defines that anyone spending over $10,000 is "Gold" because that represents their top-tier VIP customers based on historical business rules.
* **Technical Bucketing:** Data-driven and mathematical. Examples include using quantiles (percentiles) or machine learning (like `Bucketizer`) to split data evenly into buckets (e.g., Top 33%, Middle 33%, Bottom 33%) regardless of what the underlying dollar amount is.

**3. When would fixed thresholds fail?**
Fixed thresholds (e.g., `spend > 10000`) fail when the underlying data distribution changes over time, or when applied to a skewed dataset (like our sample dataset where the max spend was only a few hundred dollars). In our test above, using a strict $10,000 threshold resulted in exactly zero "Gold" or "Silver" customers because the arbitrary fixed numbers didn't match the reality of the data.

**4. How does quantile-based segmentation differ from fixed rules?**
Quantile-based segmentation dynamically adapts to the data. Instead of saying "Gold is > $10,000", it says "Gold is the top 33% of spenders, whatever that amount happens to be". If the overall sales double next month due to a promotion, the top 33% boundary will automatically shift upward. Fixed rules remain static and don't adapt to changing distributions.

**5. Which method would you use in real-world projects?**
In the real world, the best method depends entirely on the use case:
* If a business has rigid rules (e.g., "Free shipping applies to carts over $50"), you must use **fixed conditional logic**.
* If you want to dynamically find your "Top 10% best customers" for a loyalty program, or if you're preparing features for a Machine Learning model, **quantile-based segmentation** is vastly superior because it scales perfectly with changing data distributions.
* Often, projects use a mix: *Quantiles* to explore the data, which then inform the *Fixed Thresholds* agreed upon by the business.